In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your_api_key'

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.tools import tool

from langgraph.prebuilt import ToolNode # 툴에다가 외부 API가 아니라 가상의 함수 선언

In [ ]:
@tool
def get_weather(location: str):
    """Call to get the weather"""
    if location in ["서울", "인천"]:
        return "It's 60 degrees and foggy."
    else:
        return "It's 90 degress and sunny."

@tool
def get_coolest_cities():
    """Get a list of coolest cities"""
    return "서울, 고성"

In [ ]:
tools = [get_weather, get_coolest_cities]
tool_node = ToolNode(tools)

In [ ]:
from langchain_openai import ChatOpenAI

model_with_tools = ChatOpenAI(
    model="gpt-4o-mini", temperature=0
).bind_tools(tools)

In [ ]:
model_with_tools.invoke("서울 날씨는 어떄?").tool_calls

In [ ]:
model_with_tools.invoke("한국에서 가장 추운 도시는?").tool_calls

In [ ]:
tool_node.invoke({"messages": [model_with_tools.invoke("서울 날씨는 어떄?")]})

In [ ]:
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph, MessagesState

def should_continue(state: MessagesState) -> Literal["tools", END]:
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

def call_model(state: MessagesState):
    messages = state['messages']
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

workflow = StateGraph(MessagesState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")

workflow.add_conditional_edges(
    "agent",
    should_continue, # 입력값에 따라 분기가 갈라짐.
)

workflow.add_edge("tools", "agent") # tools agent에게 값을 던져줌. -> 이어짐

app = workflow.compile()


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
final_state = app.invoke(
    {"messages": [HumanMessage(content="서울의 날씨는 어떄?")]}
)
final_state["messages"][-1].content

In [ ]:
for chunk in app.stream(
    {"messages": [("human", "가장 추운 도시의 날씨는 어때?")]},
    stream_mode="values",
): 
    chunk["messages"][-1].pretty_print()